# From pynq import Overlay

In [62]:
from pynq import Overlay
from pynq.lib import AxiGPIO
import time


In [63]:
overlay = Overlay('design_1.bit')

In [66]:
overlay?

In [72]:
snnip = overlay.SNN_IP_0

In [73]:
class SNNHandler:
    OFFSET = 4  # Byte offset for each 32-bit word

    def __init__(self, handler, num_timesteps=50, num_classes=3):
        self.handler = handler
        self.num_timesteps = num_timesteps
        self.num_classes = num_classes

        self.START_ADDR        = num_timesteps * self.OFFSET       # 200
        self.RESET_ADDR        = (num_timesteps + 1) * self.OFFSET # 204
        self.SAMPLE_READY_ADDR = (num_timesteps + 2) * self.OFFSET # 208
        self.READY_ADDR        = (num_timesteps + 3) * self.OFFSET # 212

    def start(self):
        self.handler.write(self.START_ADDR, 1)
        self.handler.write(self.START_ADDR, 0)

    def reset(self):
        self.handler.write(self.RESET_ADDR, 1)
        self.handler.write(self.RESET_ADDR, 0)

    def sample_ready(self, value): #either 1 or 0
        self.handler.write(self.SAMPLE_READY_ADDR, value)

    def ready(self):
        return self.handler.read(self.READY_ADDR)

    def write_spikes(self, data):
        if len(data) != self.num_timesteps:
            raise ValueError(f"Expected {self.num_timesteps} elements.")
        for x in range(self.num_timesteps):
            self.handler.write(x * self.OFFSET, data[x])

    def read_spikes(self):
        return [self.handler.read(x * self.OFFSET) for x in range(self.num_timesteps)]

    def count_spikes_per_class(self, data=None):
        if data is None:
            data = self.read_spikes()
        if len(data) != self.num_timesteps:
            raise ValueError(f"Expected {self.num_timesteps} elements.")

        class_spike_counts = [0] * self.num_classes
        for word in data:
            for bit_pos in range(self.num_classes):
                if (word >> bit_pos) & 1:
                    class_spike_counts[bit_pos] += 1

        return class_spike_counts


In [74]:
import random

dummy_data = [random.getrandbits(32) for _ in range(50)]

In [75]:
snn = SNNHandler(snnip)

snn.reset()
snn.write_spikes(dummy_data)
snn.sample_ready(1)
snn.start()

# Wait
while not snn.ready():
    pass

output_data = snn.read_spikes()
counts = snn.count_spikes_per_class(output_data)
print("Spikes per class:", counts)

Spikes per class: [0, 11, 13]


# Test With Dataset

In [57]:
import numpy as np

In [58]:
x_test = np.load('x_test_spikes.npy')  # Replace with your filename
y_test = np.load('y_test_spikes.npy')  # Replace with your filename

In [160]:
trydata = x_test[0,0,:]

In [161]:
print(trydata)

[ 0.  0.  0.  0.  0.  1.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.]


In [170]:
def row2uint32(row):
    # Ensure binary format (0 or 1)
    bits = [int(b) for b in row]
    # LSB = row[-1], so reverse bits
    bitstring = ''.join(str(b) for b in bits[::-1])
    return int(bitstring, 2)

In [198]:
data = row2uint32(row)
print(f"Packed: {data}")
print(f"Binary : {format(data, '032b')}")

Packed: 67109920
Binary : 00000100000000000000010000100000


In [216]:
def test_sample(x_test, y_test, handler, sample_index=0, num_classes=3):
    from time import sleep

    snn = SNNHandler(handler, num_timesteps=50, num_classes=num_classes)

    # 1. Extract one sample (shape: [50, 32])
    sample = x_test[:, sample_index, :]  # (timesteps, 32)

    # 2. Pack 32-bit binary vectors into 32-bit uint values
    input_data = [row2uint32(row) for row in sample]

    # 3. Reset and write
    snn.reset()
    snn.write_spikes(input_data)
    snn.sample_ready(1)
    snn.start()

    # 4. Wait for processing to complete
    while not snn.ready():
        pass

    # 5. Read and decode
    output_data = snn.read_spikes()
    spike_counts = snn.count_spikes_per_class(output_data)

    # 6. Predict class
    predicted_class = np.argmax(spike_counts)
    true_class = y_test[sample_index]

    # 7. Display results
    print(f"\nSample {sample_index}")
    print(f"True label      : {true_class}")
    print(f"Predicted class : {predicted_class}")
    print(f"Spike counts    : {spike_counts}")
    print(f"Output bits     :")
    for i, val in enumerate(output_data):
        print(f"T={i:02d}: {format(val, '032b')}")

    return predicted_class, true_class, spike_counts


In [217]:
pred, truth, spikes = test_sample(x_test, y_test, snnip, sample_index=6)


Sample 6
True label      : 2
Predicted class : 2
Spike counts    : [0, 3, 8]
Output bits     :
T=00: 00000000000000000000000000000000
T=01: 00000000000000000000000000000000
T=02: 00000000000000000000000000000000
T=03: 00000000000000000000000000000000
T=04: 00000000000000000000000000000000
T=05: 00000000000000000000000000000000
T=06: 00000000000000000000000000000100
T=07: 00000000000000000000000000000010
T=08: 00000000000000000000000000000000
T=09: 00000000000000000000000000000000
T=10: 00000000000000000000000000000000
T=11: 00000000000000000000000000000000
T=12: 00000000000000000000000000000000
T=13: 00000000000000000000000000000000
T=14: 00000000000000000000000000000100
T=15: 00000000000000000000000000000000
T=16: 00000000000000000000000000000000
T=17: 00000000000000000000000000000000
T=18: 00000000000000000000000000000000
T=19: 00000000000000000000000000000100
T=20: 00000000000000000000000000000000
T=21: 00000000000000000000000000000010
T=22: 00000000000000000000000000000000
T=23: 0

In [220]:
def compute_confusion_matrix(predictions, labels, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for true, pred in zip(labels, predictions):
        cm[true, pred] += 1
    return cm

def print_confusion_matrix(cm):
    num_classes = cm.shape[0]
    print("\nConfusion Matrix:")
    print(" " * 10 + "Predicted")
    header = "True ▼ | " + " ".join([f"{i:^5}" for i in range(num_classes)])
    print(header)
    print("-" * len(header))
    for i, row in enumerate(cm):
        row_str = " ".join([f"{val:^5}" for val in row])
        print(f"{i:^8} | {row_str}")


def test_samples(x_test, y_test, handler, num_classes=3, verbose=False):
    snn = SNNHandler(handler, num_timesteps=50, num_classes=num_classes)

    num_samples = x_test.shape[1]
    correct = 0
    predictions = []

    for sample_index in range(num_samples):
        sample = x_test[:, sample_index, :]  # shape (50, 32)
        input_data = [row2uint32(row) for row in sample]

        # Run inference
        snn.reset()
        snn.write_spikes(input_data)
        snn.sample_ready(1)
        snn.start()

        # Wait until SNN signals it's done
        while not snn.ready():
            pass

        output_data = snn.read_spikes()
        spike_counts = snn.count_spikes_per_class(output_data)

        predicted_class = np.argmax(spike_counts)
        true_class = y_test[sample_index]
        predictions.append(predicted_class)

        if predicted_class == true_class:
            correct += 1
        

    accuracy = correct / num_samples
    print(f"\n Accuracy: {accuracy*100:.2f}% ({correct}/{num_samples})")
    
    cm = compute_confusion_matrix(predictions, y_test, num_classes)
    print_confusion_matrix(cm)

In [221]:
test_samples(x_test, y_test, snnip)


 Accuracy: 80.75% (3012/3730)

Confusion Matrix:
          Predicted
True ▼ |   0     1     2  
--------------------------
   0     |  321   160    9  
   1     |  115  1365   118 
   2     |  12    304  1326 
